In [ ]:
# Check:: the right venv and the data files exist
import sys
from pathlib import Path

print("Python:", sys.executable)
print()

from gnome.data import ENERGIES_CSV, STRUCTURES_JSON
print("Energies CSV:    ", ENERGIES_CSV, "exists:", ENERGIES_CSV.exists())
print("Structures JSON: ", STRUCTURES_JSON, "exists:", STRUCTURES_JSON.exists())

In [ ]:
import os
from pathlib import Path

print("CWD:", os.getcwd())
print("Resolved data dir:", Path("./data").resolve())
print("Resolved data dir exists:", Path("./data").resolve().exists())

In [5]:
# Imports for graph construction
import numpy as np
import torch
from torch_geometric.data import Data
from pymatgen.core import Structure


# GNoME paper specifies 4.0 Å cutoff for the structural property GNN.
EDGE_CUTOFF_ANGSTROMS = 4.0

# 100 covers all elements up to Fermium with margin for periodic-table
# edge cases beyond what MP contains.
NUM_ELEMENTS = 100

# Gaussian basis size. 64 sits between CGCNN's 41 and MEGNet's 100,
# matched to a 256-dim hidden layer downstream.
NUM_RBF = 64

# r_max exceeds EDGE_CUTOFF so the last Gaussian center is not at the
# truncation boundary. r_min=0 covers minimum bond length.
RBF_R_MIN = 0.0
RBF_R_MAX = 5.0


def expand_gaussians(distances: torch.Tensor, num_gaussians: int = NUM_RBF,
                      r_min: float = RBF_R_MIN,
                      r_max: float = RBF_R_MAX) -> torch.Tensor:
    """Expand scalar distances into a basis of equally-spaced Gaussians.

    Input shape:  (N,)
    Output shape: (N, num_gaussians)

    A distributed bond-length representation gives downstream layers
    independent channels per length range, analogous to RGB channels
    in vision.
    """
    centers = torch.linspace(r_min, r_max, num_gaussians,
                              device=distances.device)
    sigma = (r_max - r_min) / (num_gaussians - 1)
    diff = distances.unsqueeze(-1) - centers          # (N, 1) - (G,) → (N, G)
    return torch.exp(-0.5 * (diff / sigma) ** 2)


def structure_to_graph(structure: Structure,
                        e_form_per_atom: float) -> Data | None:
    """Convert a pymatgen Structure to a PyG Data object.

    Returns None for isolated atoms or structures with no edges within
    cutoff. Caller filters Nones from the dataset.
    """
    # One-hot encoding of atomic number Z in [1, NUM_ELEMENTS], indexed Z-1.
    Z = np.array([site.specie.Z for site in structure])
    if Z.max() > NUM_ELEMENTS:
        return None
    x = torch.zeros(len(Z), NUM_ELEMENTS, dtype=torch.float32)
    x[torch.arange(len(Z)), torch.from_numpy(Z) - 1] = 1.0

    # Periodic-image-aware neighbor search. Returns four parallel arrays:
    #   centers[i]   - source atom index
    #   neighbors[i] - destination atom index within the unit cell
    #   offsets[i]   - periodic-image offset (3-vector)
    #   distances[i] - distance in Å
    centers, neighbors, offsets, distances = structure.get_neighbor_list(
        r=EDGE_CUTOFF_ANGSTROMS)
    if len(centers) == 0:
        return None

    # PyG convention: edge_index is (2, E), int64.
    edge_index = torch.from_numpy(
        np.stack([centers, neighbors])).long()
    edge_attr = expand_gaussians(torch.from_numpy(distances).float())

    pos = torch.from_numpy(structure.cart_coords).float()
    y = torch.tensor([e_form_per_atom], dtype=torch.float32)

    # num_atoms is preserved separately. PyG batching concatenates x
    # across graphs, making x.size(0) ambiguous post-batch.
    return Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        pos=pos,
        y=y,
        num_atoms=torch.tensor([len(Z)], dtype=torch.long),
    )


print("Defined: expand_gaussians, structure_to_graph")
print(f"Cutoff: {EDGE_CUTOFF_ANGSTROMS} Å, RBF channels: {NUM_RBF}")

Defined: expand_gaussians, structure_to_graph
Cutoff: 4.0 Å, RBF channels: 64


**smoke-test for definitions**

1. Single element

In [6]:
# Single-structure smoke test.
# load_mp_data is expensive (~minutes); iter_mp_entries streams,
# yielding entries one at a time. islice grabs the first one cheaply.
from itertools import islice
from gnome.data import iter_mp_entries

first = next(islice(iter_mp_entries(), 1))

print(f"material_id:    {first['material_id']}")
print(f"formula:        {first['formula_pretty']}")
print(f"e_form/atom:    {first['e_form_per_atom']:+.4f} eV/atom")
print(f"n_atoms:        {len(first['structure'])}")
print(f"lattice volume: {first['structure'].lattice.volume:.2f} Å³")
print()

graph = structure_to_graph(first['structure'], first['e_form_per_atom'])

print("Graph tensors:")
print(f"  x          shape={tuple(graph.x.shape)}        dtype={graph.x.dtype}")
print(f"  edge_index shape={tuple(graph.edge_index.shape)}     dtype={graph.edge_index.dtype}")
print(f"  edge_attr  shape={tuple(graph.edge_attr.shape)}    dtype={graph.edge_attr.dtype}")
print(f"  pos        shape={tuple(graph.pos.shape)}          dtype={graph.pos.dtype}")
print(f"  y          shape={tuple(graph.y.shape)}              value={graph.y.item():+.4f}")
print(f"  num_atoms  value={graph.num_atoms.item()}")
print()

# Sanity-check derived quantities.
n_atoms = graph.num_atoms.item()
n_edges = graph.edge_index.shape[1]
avg_deg = n_edges / n_atoms
print(f"Average degree (edges per atom): {avg_deg:.1f}")
print(f"Active one-hot indices (Z-1):   {torch.where(graph.x.sum(0) > 0)[0].tolist()}")
print(f"edge_attr sample (first edge):  {graph.edge_attr[0, :5].tolist()}  (first 5 of {NUM_RBF} channels)")

e:\Projects\gnome-repro-structural\.venv\Lib\site-packages\pymatgen\core\composition.py:1398: UserWarning: No Pauling electronegativity for Ar. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  return sorted(sym, key=lambda x: [float("inf") if math.isnan(e_neg := get_el_sp(x).X) else e_neg, x])


material_id:    mp-23155
formula:        Ar
e_form/atom:    +0.0000 eV/atom
n_atoms:        1
lattice volume: 44.87 Å³

Graph tensors:
  x          shape=(1, 100)        dtype=torch.float32
  edge_index shape=(2, 12)     dtype=torch.int64
  edge_attr  shape=(12, 64)    dtype=torch.float32
  pos        shape=(1, 3)          dtype=torch.float32
  y          shape=(1,)              value=+0.0000
  num_atoms  value=1

Average degree (edges per atom): 12.0
Active one-hot indices (Z-1):   [17]
edge_attr sample (first edge):  [0.0, 0.0, 0.0, 0.0, 0.0]  (first 5 of 64 channels)


In [7]:
# Survey the first 10 MP entries to see graph behavior across structure types.
# islice avoids loading the full dataset.
from itertools import islice
from collections import Counter

samples = list(islice(iter_mp_entries(), 50))

print(f"{'idx':>3}  {'material':>12}  {'formula':>10}  {'n_at':>4}  {'edges':>5}  {'avg_deg':>7}  {'e_form':>7}")
print("-" * 70)

for i, entry in enumerate(samples):
    g = structure_to_graph(entry['structure'], entry['e_form_per_atom'])
    if g is None:
        print(f"{i:>3}  {entry['material_id']:>12}  {entry['formula_pretty']:>10}  SKIPPED (no edges or bad Z)")
        continue
    n_atoms = g.num_atoms.item()
    n_edges = g.edge_index.shape[1]
    print(f"{i:>3}  {entry['material_id']:>12}  {entry['formula_pretty']:>10}  "
          f"{n_atoms:>4}  {n_edges:>5}  {n_edges/n_atoms:>7.2f}  "
          f"{entry['e_form_per_atom']:>+7.3f}")

# Pick the most chemically interesting one (most unique elements)
# for deeper inspection.
def n_unique_elements(structure):
    return len({s.specie.Z for s in structure})

interesting_idx = max(range(len(samples)),
                       key=lambda i: n_unique_elements(samples[i]['structure']))
chosen = samples[interesting_idx]
print(f"\nDeeper look at idx {interesting_idx}: {chosen['material_id']} ({chosen['formula_pretty']})")

g = structure_to_graph(chosen['structure'], chosen['e_form_per_atom'])

# Element distribution within the structure
Z_list = [s.specie.Z for s in chosen['structure']]
element_counts = Counter(s.specie.symbol for s in chosen['structure'])
print(f"  composition: {dict(element_counts)}")
print(f"  unique Zs:   {sorted(set(Z_list))}")

# Verify one-hot encoding matches composition
active_indices = torch.where(g.x.sum(0) > 0)[0].tolist()
expected_indices = sorted(z - 1 for z in set(Z_list))
print(f"  one-hot indices in x:        {active_indices}")
print(f"  expected (Z-1 per species):  {expected_indices}")
print(f"  match: {active_indices == expected_indices}")

# Edge-length distribution: edge_attr peaks should align with bond lengths
edge_lengths_recovered = []
for i in range(min(5, g.edge_index.shape[1])):
    # Find which Gaussian channel has the strongest response → recover distance
    channel_responses = g.edge_attr[i]
    peak_channel = channel_responses.argmax().item()
    peak_distance = RBF_R_MIN + peak_channel * (RBF_R_MAX - RBF_R_MIN) / (NUM_RBF - 1)
    edge_lengths_recovered.append(peak_distance)
print(f"  first 5 edge bond lengths (recovered from Gaussian peaks):")
for d in edge_lengths_recovered:
    print(f"    {d:.2f} Å")

e:\Projects\gnome-repro-structural\.venv\Lib\site-packages\pymatgen\core\composition.py:1398: UserWarning: No Pauling electronegativity for Ne. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  return sorted(sym, key=lambda x: [float("inf") if math.isnan(e_neg := get_el_sp(x).X) else e_neg, x])


idx      material     formula  n_at  edges  avg_deg   e_form
----------------------------------------------------------------------
  0      mp-23155          Ar     1     12    12.00   +0.000
  1    mp-1246134          Ni     1     20    20.00   +0.410
  2    mp-1182070          Bi     4     24     6.00   +0.009
  3     mp-569358          Bi     2     12     6.00   +0.550
  4    mp-1078637          Bi     8     84    10.50   +0.051
  5     mp-567597          Bi     1      6     6.00   +0.000
  6     mp-568610          Bi     1     14    14.00   +0.090
  7      mp-11534          Np     8    120    15.00   +0.000
  8      mp-22912        NpBi     2     12     6.00   +0.064
  9    mp-1183057          Ac     3     12     4.00   +0.016
 10    mp-1183069          Ac     3     18     6.00   +0.045
 11    mp-1014111          Ni     2     24    12.00   +0.620
 12      mp-10257          Ni     2     36    18.00   +0.026
 13      mp-23157          Bi     2     18     9.00   +0.001
 14    mp-1182

In [ ]:
# Smoke instantiation: create the model with a dummy avg_adjacency, count params.
import importlib
if "gnome.model" in dir():
    importlib.reload(gnome.model)

from gnome.model import GNoMEStructural

model = GNoMEStructural(avg_adjacency=12.0)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model instantiated: {n_params:,} parameters")
print(f"Components:")
for name, mod in model.named_children():
    sub_params = sum(p.numel() for p in mod.parameters())
    print(f"  {name:15s}  {sub_params:>9,} params")

**Load**

In [ ]:
# Load full dataset into memory (one-time cost: ~minutes due to JSON parsing)
# Reload-friendly: rerun this cell only if entries is not yet defined,
# to avoid paying the parse cost again.

if "entries" not in globals():
    from gnome.data import load_mp_data
    entries = load_mp_data(verbose=True)
    print(f"\nLoaded {len(entries)} entries.")
else:
    print(f"Already loaded: {len(entries)} entries (skipping reload).")